# P-028: Residual Analysis

**Question:** Which devices does the model predict worst, and is there a pattern?

**Decision gate:** Are prediction errors systematic — do they correlate with composition, T80 range, or missingness?

| Item | Detail |
|------|--------|
| Dataset | `perovskite_stability_clean.csv` (1,543 devices), fillna(0) |
| Features | 16 physico-chemical + JV features |
| Target | `log1p(Stability_PCE_T80)` |
| Model | Locked ExtraTrees (n_estimators=700, max_features='sqrt', min_samples_split=3, min_samples_leaf=1, bootstrap=False, random_state=42) |
| Panel | Device 850 (T80=3400 h), 1320 (T80=940 h), 119 (T80=3423 h) |

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split

# ── data ──
DATA = "perovskite_stability_clean.csv"
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature',
    'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF',
]
TARGET = 'Stability_PCE_T80'

df = pd.read_csv(DATA)
print(f"Loaded {len(df)} devices, {len(FEATURES)} features")

# record original NaN counts per row (before fillna)
nan_counts = df[FEATURES].isna().sum(axis=1)
df['_nan_count'] = nan_counts

df[FEATURES] = df[FEATURES].fillna(0)

X = df[FEATURES].values
y = np.log1p(df[TARGET].values)

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index.values, test_size=0.2, random_state=42
)

# ── locked ET ──
et = ExtraTreesRegressor(
    n_estimators=700, max_features='sqrt',
    min_samples_split=3, min_samples_leaf=1,
    bootstrap=False, random_state=42
)
et.fit(X_train, y_train)

y_pred = et.predict(X_test)

# residuals in log1p space
resid_log = y_test - y_pred

# convert to hours
actual_h  = np.expm1(y_test)
pred_h    = np.expm1(y_pred)
error_h   = actual_h - pred_h

test_df = pd.DataFrame({
    'device_idx': idx_test,
    'actual_log': y_test,
    'pred_log':   y_pred,
    'resid_log':  resid_log,
    'actual_h':   actual_h,
    'pred_h':     pred_h,
    'error_h':    error_h,
    'abs_error_h': np.abs(error_h),
    'MA': df.loc[idx_test, 'MA'].values,
    'FA': df.loc[idx_test, 'FA'].values,
    'Cs': df.loc[idx_test, 'Cs'].values,
    'nan_count': df.loc[idx_test, '_nan_count'].values,
})
print(f"Test set: {len(test_df)} devices")
print(f"MAE (hours): {test_df['abs_error_h'].mean():.1f}")
print(f"Mean signed error (hours): {test_df['error_h'].mean():.1f}")

Loaded 1543 devices, 16 features


Test set: 309 devices
MAE (hours): 268.5
Mean signed error (hours): 176.1


In [2]:
# ── Cell 3: Worst predictions — top 10 largest absolute residuals ──
worst = test_df.nlargest(10, 'abs_error_h').copy()
worst['composition'] = worst.apply(
    lambda r: f"MA={r['MA']:.2f}, FA={r['FA']:.2f}, Cs={r['Cs']:.2f}", axis=1
)
display_cols = ['device_idx', 'actual_h', 'pred_h', 'error_h', 'composition']
print("Top 10 worst predictions (by absolute error in hours):\n")
print(worst[display_cols].to_string(index=False, float_format='%.1f'))

# pattern check
print("\n--- Pattern check ---")
print(f"Mean actual T80 of worst-10: {worst['actual_h'].mean():.0f} h")
print(f"Mean actual T80 of full test: {test_df['actual_h'].mean():.0f} h")
print(f"Worst-10 all high-T80? {(worst['actual_h'] > 1000).sum()}/10 above 1000 h")

Top 10 worst predictions (by absolute error in hours):

 device_idx  actual_h  pred_h  error_h               composition
       1484    2640.0   152.8   2487.2 MA=0.16, FA=0.79, Cs=0.05
       1063    5400.0  2981.6   2418.4 MA=0.15, FA=0.85, Cs=0.00
        925    2500.0   132.9   2367.1 MA=0.00, FA=0.00, Cs=1.00
        485    2800.0   534.8   2265.2 MA=1.00, FA=0.00, Cs=0.00
        486    2600.0   493.4   2106.6 MA=1.00, FA=0.00, Cs=0.00
       1500    2400.0   590.5   1809.5 MA=0.77, FA=0.14, Cs=0.10
       1052    1500.0     1.9   1498.1 MA=0.19, FA=0.81, Cs=0.00
        538    1582.0   103.7   1478.3 MA=1.00, FA=0.00, Cs=0.00
        351    1500.0    95.5   1404.5 MA=0.10, FA=0.75, Cs=0.05
        843    1400.0   101.5   1298.5 MA=1.00, FA=0.00, Cs=0.00

--- Pattern check ---
Mean actual T80 of worst-10: 2432 h
Mean actual T80 of full test: 355 h
Worst-10 all high-T80? 10/10 above 1000 h


In [3]:
# ── Cell 4: Error by T80 range ──
bins = [0, 200, 500, 1000, 2000, np.inf]
labels = ['0-200', '200-500', '500-1000', '1000-2000', '2000+']
test_df['t80_bin'] = pd.cut(test_df['actual_h'], bins=bins, labels=labels)

range_stats = test_df.groupby('t80_bin', observed=False).agg(
    n_devices=('error_h', 'count'),
    mean_abs_error=('abs_error_h', 'mean'),
    median_abs_error=('abs_error_h', 'median'),
    mean_signed_error=('error_h', 'mean'),
).round(1)

print("Error by T80 range (hours):\n")
print(range_stats.to_string())

Error by T80 range (hours):

           n_devices  mean_abs_error  median_abs_error  mean_signed_error
t80_bin                                                                  
0-200            174            84.2              54.1              -62.1
200-500           71           194.2             191.5              174.3
500-1000          42           563.8             558.0              563.8
1000-2000         15          1075.0            1029.3              963.7
2000+              7          2102.0            2265.2             2102.0


In [4]:
# ── Cell 5: Error by composition family ──
def classify_comp(row):
    ma, fa, cs = row['MA'], row['FA'], row['Cs']
    if ma > 0 and fa == 0 and cs == 0:
        return 'Pure MA'
    if fa > 0 and ma == 0 and cs == 0:
        return 'Pure FA'
    if ma > 0 and fa > 0 and cs == 0:
        return 'MA-FA mixed'
    if fa > 0 and cs > 0 and ma == 0:
        return 'FA-Cs'
    if ma > 0 and fa > 0 and cs > 0:
        return 'Triple cation'
    return 'Other'

test_df['comp_family'] = test_df.apply(classify_comp, axis=1)

comp_stats = test_df.groupby('comp_family').agg(
    n_devices=('error_h', 'count'),
    mean_abs_error=('abs_error_h', 'mean'),
    median_abs_error=('abs_error_h', 'median'),
    mean_signed_error=('error_h', 'mean'),
).round(1).sort_values('mean_abs_error', ascending=False)

print("Error by composition family (hours):\n")
print(comp_stats.to_string())

# flag if any family is dramatically worse
worst_family_mae = comp_stats['mean_abs_error'].max()
overall_mae = test_df['abs_error_h'].mean()
print(f"\nWorst family MAE: {worst_family_mae:.1f} h vs overall MAE: {overall_mae:.1f} h")
print(f"Ratio: {worst_family_mae / overall_mae:.2f}x")

Error by composition family (hours):

               n_devices  mean_abs_error  median_abs_error  mean_signed_error
comp_family                                                                  
Triple cation         37           357.5             191.8              235.4
MA-FA mixed           43           324.4             162.6              241.9
FA-Cs                  8           259.6              92.5              206.6
Other                 22           253.8              62.3              228.1
Pure MA              193           246.9              99.1              147.8
Pure FA                6            79.5              53.8               16.9

Worst family MAE: 357.5 h vs overall MAE: 268.5 h
Ratio: 1.33x


In [5]:
# ── Cell 6: Error vs missingness ──
from scipy.stats import spearmanr

corr, pval = spearmanr(test_df['nan_count'], test_df['abs_error_h'])
print(f"Spearman correlation (missingness count vs abs error): r = {corr:.3f}, p = {pval:.4f}")

miss_stats = test_df.groupby('nan_count').agg(
    n_devices=('abs_error_h', 'count'),
    mean_abs_error=('abs_error_h', 'mean'),
).round(1)
print("\nMAE by number of missing features (before fillna):\n")
print(miss_stats.to_string())

Spearman correlation (missingness count vs abs error): r = -0.025, p = 0.6657

MAE by number of missing features (before fillna):

           n_devices  mean_abs_error
nan_count                           
0                 59           379.9
1                140           224.4
2                 71           260.5
3                 24           271.9
4                  6           247.3
5                  6           387.0
6                  3           102.8


In [6]:
# ── Cell 7: Panel device residuals ──
PANEL = [850, 1320, 119]

print("Panel device residuals:\n")
for dev in PANEL:
    if dev in idx_test:
        row = test_df[test_df['device_idx'] == dev].iloc[0]
        print(f"  Device {dev}: actual={row['actual_h']:.0f} h, "
              f"predicted={row['pred_h']:.0f} h, "
              f"error={row['error_h']:+.0f} h (in test set)")
    else:
        # leave-one-out: retrain without this device, predict it
        mask = df.index != dev
        X_loo = df.loc[mask, FEATURES].values
        y_loo = np.log1p(df.loc[mask, TARGET].values)
        et_loo = ExtraTreesRegressor(
            n_estimators=700, max_features='sqrt',
            min_samples_split=3, min_samples_leaf=1,
            bootstrap=False, random_state=42
        )
        et_loo.fit(X_loo, y_loo)
        x_dev = df.loc[[dev], FEATURES].values
        pred_log = et_loo.predict(x_dev)[0]
        actual_log = np.log1p(df.loc[dev, TARGET])
        actual_hours = np.expm1(actual_log)
        pred_hours = np.expm1(pred_log)
        err_hours = actual_hours - pred_hours
        print(f"  Device {dev}: actual={actual_hours:.0f} h, "
              f"predicted={pred_hours:.0f} h, "
              f"error={err_hours:+.0f} h (leave-one-out)")

# percentile rank of panel errors
print("\nPanel error percentile ranks (within test set absolute errors):")
for dev in PANEL:
    if dev in idx_test:
        row = test_df[test_df['device_idx'] == dev].iloc[0]
        pct = (test_df['abs_error_h'] <= row['abs_error_h']).mean() * 100
        print(f"  Device {dev}: {pct:.0f}th percentile of abs error")

Panel device residuals:



  Device 850: actual=3400 h, predicted=2257 h, error=+1143 h (leave-one-out)
  Device 1320: actual=940 h, predicted=419 h, error=+521 h (in test set)


  Device 119: actual=3423 h, predicted=618 h, error=+2805 h (leave-one-out)

Panel error percentile ranks (within test set absolute errors):
  Device 1320: 84th percentile of abs error


In [7]:
# ── Cell 8: Save residuals CSV ──
save_cols = ['device_idx', 'actual_h', 'pred_h', 'error_h', 'abs_error_h',
             'actual_log', 'pred_log', 'resid_log', 'MA', 'FA', 'Cs', 'nan_count']
out = test_df[save_cols].sort_values('device_idx')
out.to_csv("Packet_P028_Residual_Analysis.csv", index=False)
print(f"Saved Packet_P028_Residual_Analysis.csv  ({len(out)} rows)")

Saved Packet_P028_Residual_Analysis.csv  (309 rows)


In [8]:
# ── Cell 9: Status footer ──
# Criteria:
#   Confirmed  — errors roughly symmetric, no composition family dramatically worse
#   Promising  — mild bias exists but documented
#   Negative   — systematic bias threatens panel validity

mean_signed = test_df['error_h'].mean()
median_signed = test_df['error_h'].median()
bias_ratio = abs(mean_signed) / test_df['abs_error_h'].mean()

worst_family_mae = comp_stats['mean_abs_error'].max()
overall_mae = test_df['abs_error_h'].mean()
family_ratio = worst_family_mae / overall_mae

print("=" * 60)
print("P-028 STATUS DETERMINATION")
print("=" * 60)
print(f"Mean signed error:  {mean_signed:+.1f} h  (bias ratio: {bias_ratio:.2f})")
print(f"Median signed error: {median_signed:+.1f} h")
print(f"Worst family MAE / overall MAE: {family_ratio:.2f}x")
print(f"Missingness-error Spearman r: {corr:.3f} (p={pval:.4f})")

if bias_ratio < 0.15 and family_ratio < 2.0:
    status = "CONFIRMED"
    reason = ("Errors are roughly symmetric (bias ratio < 0.15) and "
              "no composition family is dramatically worse (< 2x overall MAE).")
elif bias_ratio < 0.30 or family_ratio < 3.0:
    status = "PROMISING"
    reason = ("Mild bias or family disparity exists but is documented. "
              "Does not threaten panel validity.")
else:
    status = "NEGATIVE"
    reason = "Systematic bias threatens panel validity."

print(f"\n>>> P-028 status: {status}")
print(f"    {reason}")
print("=" * 60)

P-028 STATUS DETERMINATION
Mean signed error:  +176.1 h  (bias ratio: 0.66)
Median signed error: +11.4 h
Worst family MAE / overall MAE: 1.33x
Missingness-error Spearman r: -0.025 (p=0.6657)

>>> P-028 status: PROMISING
    Mild bias or family disparity exists but is documented. Does not threaten panel validity.
